1. Preparación y carga de datos

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report

# Carga de datos
df = pd.read_csv("vino.csv")
columnas = ["acidez_fija","acidez_volatil","acido_citrico","azucar_residual","cloruros",
            "dioxido_azufre_libre","dioxido_azufre_total","densidad","ph","sulfatos","alcohol","calidad"]
df.columns = columnas

# Transformación binaria
df['calidad_binaria'] = np.where(df['calidad'] >= 6, 'Alta', 'Baja')
df = df.drop(columns=['calidad'])

print(df.describe())

2. Análisis Estadístico y Gráficas

In [ ]:
# ANOVA equivalente
from statsmodels.formula.api import ols
import statsmodels.api as sm

for col in df.columns.drop('calidad_binaria'):
    model = ols(f'{col} ~ calidad_binaria', data=df).fit()
    anova_table = sm.stats.anova_lm(model, typ=2)
    print(f"\nANOVA para {col}:\n", anova_table)

# Boxplots
fig, axes = plt.subplots(3, 4, figsize=(15, 12))
for i, col in enumerate(df.columns.drop('calidad_binaria')):
    sns.boxplot(x='calidad_binaria', y=col, data=df, ax=axes.flatten()[i])
plt.tight_layout()
plt.show()

In [ ]:
3. Modelos (Árbol y Random Forest)

In [ ]:
# División 80/20
X = df.drop(columns=['calidad_binaria'])
y = df['calidad_binaria']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)

# Árbol de decisión
arbol = DecisionTreeClassifier(random_state=123)
arbol.fit(X_train, y_train)

# Random Forest
forest = RandomForestClassifier(n_estimators=500, random_state=123)
forest.fit(X_train, y_train)

# Evaluación
print("Matriz Árbol:\n", confusion_matrix(y_test, arbol.predict(X_test)))
print("Matriz Forest:\n", confusion_matrix(y_test, forest.predict(X_test)))

4. Variables Importantes

In [ ]:
importances = pd.Series(forest.feature_importances_, index=X.columns)
importances.sort_values().plot(kind='barh', color='teal')
plt.title("Variables Importantes")
plt.show()

5. Evaluación con variables reducidas

In [ ]:
# Top 4: "alcohol", "sulfatos", "acidez_volatil", "dioxido_azufre_total"
cols_top4 = ["alcohol", "sulfatos", "acidez_volatil", "dioxido_azufre_total"]
X_red = X[cols_top4]
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_red, y, test_size=0.2, random_state=123)

forest_red = RandomForestClassifier(n_estimators=500, random_state=123)
forest_red.fit(X_train_r, y_train_r)

print("Reporte con Top 4:\n", classification_report(y_test_r, forest_red.predict(X_test_r)))